In [ ]:
# hide
import numpy as np
import pyquist as pq


def adenv(a_dur: float, d_dur: float, N: int, f_s: int = 44100) -> np.ndarray:
    """A simple attack/decay envelope, shaped (N, 1)."""
    t = np.arange(N) / f_s
    env = np.interp(t, [0.0, a_dur, a_dur + d_dur], [0.0, 1.0, 0.0])
    return env[:, np.newaxis]

In [ ]:
# An instrument maps one event's kwargs to Audio. Here: an enveloped sine tone.
def sine_instrument(pitch: str, duration: float, f_s: int = 44100, **kwargs) -> pq.Audio:
    f_0 = pq.helper.pitch_to_frequency(pq.helper.pitch_name_to_pitch(pitch))
    N = int(duration * f_s)
    t = np.arange(N) / f_s
    tone = np.sin(2 * np.pi * f_0 * t)
    result = pq.Audio(tone, f_s)
    result *= adenv(0.01, duration - 0.01, N, f_s) # attack/decay so notes don't click
    return result


# A simple C-D-E-F-G melody, each note half a second long.
melody = pq.Score([
    (0.0, {"pitch": "C4", "duration": 0.5}),
    (0.5, {"pitch": "D4", "duration": 0.5}),
    (1.0, {"pitch": "E4", "duration": 0.5}),
    (1.5, {"pitch": "F4", "duration": 0.5}),
    (2.0, {"pitch": "G4", "duration": 0.5}),
])
audio = melody.render(sine_instrument)   # render calls the instrument once per event and mixes
pq.play(audio)